In [1]:
import re
import io
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

In [2]:
HEADERS = {"User-Agent": "Mozilla/5.0"}
EIA_DIESEL_URL = "https://www.eia.gov/petroleum/gasdiesel/xls/psw18vwall.xls"
EIA_GAS_URL = "https://www.eia.gov/petroleum/gasdiesel/xls/pswrgvwall.xls"

def clean_name(s):
    return re.sub(r"\s+", " ", str(s)).strip()

def load_eia(url):
    raw = requests.get(url, headers=HEADERS, timeout=60)
    raw.raise_for_status()
    sheets = pd.read_excel(io.BytesIO(raw.content), sheet_name=None, header=2)
    frames = []
    for name, df in sheets.items():
        if not name.startswith("Data"):
            continue
        df = df.rename(columns=clean_name)
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df = df.dropna(subset=["Date"]).set_index("Date")
        frames.append(df)
    return pd.concat(frames, axis=1)

diesel_all = load_eia(EIA_DIESEL_URL)
gas_all = load_eia(EIA_GAS_URL)

print("Diesel columns containing 'U.S.':")
print([c for c in diesel_all.columns if "U.S." in c])
print("Gasoline columns containing 'U.S.':")
print([c for c in gas_all.columns if "U.S." in c])

/tmp/ipykernel_23291/4198166731.py:11: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  sheets = pd.read_excel(io.BytesIO(raw.content), sheet_name=None, header=2)
/tmp/ipykernel_23291/4198166731.py:20: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  return pd.concat(frames, axis=1)
/tmp/ipykernel_23291/4198166731.py:11: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  sheets = pd.read_excel(io.BytesIO(raw.content), sheet_name=None, header=2)


Diesel columns containing 'U.S.':
['Weekly U.S. No 2 Diesel Retail Prices (Dollars per Gallon)', 'U.S. No 2 Diesel Retail Prices (Dollars per Gallon)', 'Weekly U.S. No 2 Diesel Low Sulfur (15-500 ppm) Retail Prices (Dollars per Gallon)', 'U.S. No 2 Diesel Low Sulfur (15-500 ppm) Retail Prices (Dollars per Gallon)', 'Weekly U.S. No 2 Diesel Ultra Low Sulfur (0-15 ppm) Retail Prices (Dollars per Gallon)', 'U.S. No 2 Diesel Ultra Low Sulfur (0-15 ppm) Retail Prices (Dollars per Gallon)']
Gasoline columns containing 'U.S.':
['Weekly U.S. Regular Conventional Retail Gasoline Prices (Dollars per Gallon)', 'Weekly U.S. Regular Reformulated Retail Gasoline Prices (Dollars per Gallon)', 'Weekly U.S. Regular All Formulations Retail Gasoline Prices (Dollars per Gallon)', 'Weekly U.S. Midgrade Conventional Retail Gasoline Prices (Dollars per Gallon)', 'Weekly U.S. Midgrade Reformulated Retail Gasoline Prices (Dollars per Gallon)', 'Weekly U.S. Midgrade All Formulations Retail Gasoline Prices (Doll

In [3]:
def pick_us_column(df, must_contain):
    bad = ["PADD", "Region", "Coast", "Midwest", "Rocky", "Gulf", "New England", "Atlantic"]
    cands = [c for c in df.columns
             if c.startswith("Weekly U.S.")
             and all(k.lower() in c.lower() for k in must_contain)
             and not any(b.lower() in c.lower() for b in bad)]
    print(must_contain, "->", cands)
    return cands[0]

diesel_col = pick_us_column(diesel_all, ["Diesel", "Retail"])
gas_col = pick_us_column(gas_all, ["Regular", "Retail"])

fuel = pd.DataFrame({
    "diesel": diesel_all[diesel_col],
    "gasoline": gas_all[gas_col],
}).sort_index()

fuel["week"] = fuel.index.to_period("W-SUN")
fuel = fuel.groupby("week").mean()
print(fuel.tail())
print(fuel.index.min(), fuel.index.max(), len(fuel))

['Diesel', 'Retail'] -> ['Weekly U.S. No 2 Diesel Retail Prices (Dollars per Gallon)', 'Weekly U.S. No 2 Diesel Low Sulfur (15-500 ppm) Retail Prices (Dollars per Gallon)', 'Weekly U.S. No 2 Diesel Ultra Low Sulfur (0-15 ppm) Retail Prices (Dollars per Gallon)']
['Regular', 'Retail'] -> ['Weekly U.S. Regular Conventional Retail Gasoline Prices (Dollars per Gallon)', 'Weekly U.S. Regular Reformulated Retail Gasoline Prices (Dollars per Gallon)', 'Weekly U.S. Regular All Formulations Retail Gasoline Prices (Dollars per Gallon)']
                       diesel  gasoline
week                                   
2026-10-19/2026-10-25   6.276     4.121
2026-10-26/2026-11-01   6.352     4.121
2026-11-02/2026-11-08   6.292     3.996
2026-11-09/2026-11-15   6.127     3.836
2026-11-16/2026-11-22   5.937     3.633
1990-08-20/1990-08-26 2026-11-16/2026-11-22 1892


In [7]:
import io
import requests

HEADERS = {"User-Agent": "Mozilla/5.0", "Accept": "text/csv"}

URLS = [
    "https://agtransport.usda.gov/api/views/ux59-h6jx/rows.csv?accessType=DOWNLOAD",
    "https://agtransport.usda.gov/resource/ux59-h6jx.csv?$limit=100000",
]

rates_raw = None
for url in URLS:
    r = requests.get(url, headers=HEADERS, timeout=120)
    print(r.status_code, len(r.content), "bytes |", url)
    if r.ok and len(r.content) > 100:
        rates_raw = pd.read_csv(io.BytesIO(r.content))
        break

if rates_raw is None:
    raise RuntimeError("Both endpoints returned nothing, use the manual download")

print(rates_raw.shape)
print(rates_raw.dtypes)
print(rates_raw.head())

for c in rates_raw.select_dtypes("object").columns:
    n = rates_raw[c].nunique()
    print(f"\n{c}: {n} unique")
    if n <= 25:
        print(rates_raw[c].value_counts(dropna=False))

200 207280 bytes | https://agtransport.usda.gov/api/views/ux59-h6jx/rows.csv?accessType=DOWNLOAD


EmptyDataError: No columns to parse from file

In [ ]:
date_guess = [c for c in rates_raw.columns if "date" in c.lower() or "week" in c.lower()]
rate_guess = [c for c in rates_raw.columns if "rate" in c.lower() or "mile" in c.lower()]
print("date candidates:", date_guess)
print("rate candidates:", rate_guess)

DATE_COL = date_guess[0]
RATE_COL = rate_guess[0]

# example: FILTERS = {"region": "National", "distance": "All"}
FILTERS = {}

rates = rates_raw.copy()
for col, val in FILTERS.items():
    rates = rates[rates[col] == val]

rates[DATE_COL] = pd.to_datetime(rates[DATE_COL], errors="coerce")
rates[RATE_COL] = pd.to_numeric(rates[RATE_COL], errors="coerce")
rates = rates.dropna(subset=[DATE_COL, RATE_COL])

rates["week"] = rates[DATE_COL].dt.to_period("W-SUN")
rows_per_week = rates.groupby("week").size()
print("rows per week after filters:\n", rows_per_week.describe())

truck = rates.groupby("week")[RATE_COL].mean().rename("truck_rate").to_frame()
print(truck.tail())
print(truck.index.min(), truck.index.max(), len(truck))

In [ ]:
df = truck.join(fuel, how="inner")

full_range = pd.period_range(df.index.min(), df.index.max(), freq="W-SUN")
missing_weeks = full_range.difference(df.index)
print("rows:", len(df), "| missing weeks:", len(missing_weeks))
print(missing_weeks)

df = df.reindex(full_range)
df.index.name = "week"
df[["truck_rate", "diesel", "gasoline"]] = df[["truck_rate", "diesel", "gasoline"]].interpolate(limit=2)

print(df.isna().sum())
print(df.describe())

df_ts = df.copy()
df_ts.index = df_ts.index.to_timestamp()
df_ts.to_csv("merged_weekly.csv")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
df_ts["truck_rate"].plot(ax=axes[0], title="Refrigerated truck rate ($/mile)")
df_ts["diesel"].plot(ax=axes[1], title="US diesel ($/gal)")
df_ts["gasoline"].plot(ax=axes[2], title="US regular gasoline ($/gal)")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(df_ts.index, df_ts["truck_rate"] / df_ts["truck_rate"].iloc[0], label="truck rate")
ax.plot(df_ts.index, df_ts["diesel"] / df_ts["diesel"].iloc[0], label="diesel")
ax.set_title("Indexed to first week = 1")
ax.legend()
plt.show()

In [ ]:
chg = np.log(df_ts[["truck_rate", "diesel", "gasoline"]]).diff().dropna()

lags = range(0, 13)
corr = pd.DataFrame({
    "diesel": [chg["truck_rate"].corr(chg["diesel"].shift(k)) for k in lags],
    "gasoline": [chg["truck_rate"].corr(chg["gasoline"].shift(k)) for k in lags],
}, index=list(lags))
corr.index.name = "lag_weeks"
print(corr.round(3))

corr.plot(kind="bar", figsize=(10, 4), title="Corr of weekly truck rate change with lagged fuel change")
plt.show()